# Bulk library build (continuation + exploit) — long GPU run

Builds a large, texture-diverse **continuation** + **exploit** library across many boards, on
the correct streets=3 batched GPU solver. Every generator re-writes a valid, signed pack every
`CHECKPOINT` boards, and each write is **atomic** (temp file + rename) — so if the Kaggle session
is cut off, even mid-write, the latest pack on disk stays complete and downloadable.

**Sizing.** Reference: 4 boards @ n=80, iters=600 ≈ 8–9 min on GPU float32. Per-board time scales
~`iters × n²`, so at n=120 / iters=800 that's **~6 min/board**. The default `BOARDS=48` is therefore
**~5h per generator ≈ ~10h total**, which fits comfortably inside a single Kaggle GPU session
(~12h cap) with both generators completing. Boards come from a deterministic diverse set
(max 180; `_DIVERSE_FLOPS` × 6 runouts in `demo/gen_continuation.py`).

**Want the 15–20h end?** A single Kaggle session can't exceed ~12h, so go bigger by **chunking
across sessions**: set `BOARD_START` + a distinct `SUFFIX` per session (e.g. session 1 →
`BOARD_START=0, SUFFIX='_a'`; session 2 → `BOARD_START=48, SUFFIX='_b'`). Each chunk is an
independent signed pack; download them all and they get merged locally. Re-running a chunk whose
pack already exists just overwrites it — no lost work.

Pick a **GPU** session; **Internet On**. Writes new versions `continuation_full` / `exploit_full`
(does NOT overwrite the shipped `continuation_seed` / `exploit_v1`).

In [ ]:
# Fail fast if this isn't a working GPU session.
import cupy, subprocess
ndev = cupy.cuda.runtime.getDeviceCount()
assert ndev > 0, 'No CUDA device — set the runtime Accelerator to GPU.'
print(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv'],
                     capture_output=True, text=True).stdout)
print(f'CuPy sees {ndev} GPU(s) — good to go')

In [ ]:
!rm -rf /kaggle/working/poker && git clone -q --depth 1 https://github.com/tian-chaiyaporn2/poker_offline_trainer /kaggle/working/poker
import sys; sys.path.insert(0, '/kaggle/working/poker/src')
import subprocess
print('source ready @', subprocess.run(['git','-C','/kaggle/working/poker','rev-parse','--short','HEAD'],
                                       capture_output=True, text=True).stdout.strip())

In [ ]:
# --- knobs (default ~10h across BOTH generators; fits one Kaggle GPU session) ---
#   BOARDS: diverse boards per generator (max 180).  N: combos/range.  ITERS: CFR iterations.
#   CHECKPOINT: re-write the signed (atomic) pack every K boards. Progress prints
#   `[i/total] ... stable=` per board and `wrote ... pack` at each checkpoint.
#   BOARD_START/SUFFIX: only for chunking a >12h run across sessions (see the top cell).
import subprocess, os
BOARDS, N, ITERS, CHECKPOINT = 48, 120, 800, 4
BOARD_START, SUFFIX = 0, ''
env = {**os.environ, 'PYTHONPATH': 'src'}
def gen(script, version):
    subprocess.run(['python', script, '--solver', 'gpu', '--dtype', 'float32',
                    '--boards', str(BOARDS), '--board-start', str(BOARD_START),
                    '--n', str(N), '--iters', str(ITERS),
                    '--checkpoint-every', str(CHECKPOINT), '--version', version + SUFFIX],
                   cwd='/kaggle/working/poker', env=env, check=True)
gen('demo/gen_continuation.py', 'continuation_full')
gen('demo/gen_exploit.py', 'exploit_full')

In [ ]:
# Expose whatever *full* packs exist for download (works even if the session was cut off
# mid-run — the last atomic checkpoint of each version is a valid signed pack). Globs so
# chunked runs (continuation_full_a, exploit_full_b, ...) are all picked up.
import shutil, os, glob
base = '/kaggle/working/poker/output/packs'
got = []
for p in sorted(glob.glob(os.path.join(base, 'flop_pack_*full*.db')) +
                glob.glob(os.path.join(base, 'flop_pack_*full*.db.gz')) +
                glob.glob(os.path.join(base, 'build_report_*full*.json'))):
    dst = os.path.join('/kaggle/working', os.path.basename(p))
    shutil.copy(p, dst); got.append(dst)
print('DOWNLOAD from /kaggle/working:' if got else 'No packs written yet — check the solve cell.')
for dst in got:
    print('  %-46s %d KB' % (os.path.basename(dst), os.path.getsize(dst) // 1024))